## LLM 文本增强 — 高词数版

### V1 的区别

| 参数 | V1 (旧) | V2 (新) | 原因 |
|------|:---:|:---:|------|
| `MAX_WORDS` | 50 | **200** | BGE 512 tok，200 词 ≈ 260 tok，占比 53% |
| `MAX_TOKENS` | 80 | **300** | 1 word ≈ 1.3 tok，200 words ≈ 260 tok |
| `MAX_CHARS` | 500 | **1500** | ~7 char/word × 200 = 1400 chars |

### 为什么 200 词？

120-180 词是 LLM 信息密度最高区间。200 词让 LLM 充分展开六维度但不过度，

超过 200 词 LLM 开始填 "highly acclaimed by audiences and critics alike"  这种对推荐零区分度的废话。

In [ ]:
import os, time, threading
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

DEEPSEEK_API_KEY = ""
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
MODEL = "deepseek-v4-flash"

META_CSV     = "final/item_meta_merged.csv"
OUTPUT_CSV   = "final/item_meta_llm_enhanced_v2.csv"

BATCH_SAVE   = 500
TEMPERATURE  = 0.1
MAX_TOKENS   = 300      # API 硬限制 (V1: 80)
MAX_WORDS    = 200      # prompt 引导 (V1: 50)
MAX_CHARS    = 1500     # 代码兜底 (V1: 500)
NUM_THREADS  = 20
SLEEP_RATE_LIMIT = 5

write_lock = threading.Lock()
client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url=DEEPSEEK_BASE_URL)

### Prompt (六维度展开) 

六个维度：细粒度体裁、叙事结构、风格基调、核心主题、受众情绪、跨域相似作品

拆解物品画像，同时通过字数、句式、内容边界的多重约束，输出区分度高、无营销冗余的自然语义文本

In [2]:
def build_prompt(domain, title):
    domain_name = "Movie/TV" if domain == 0 else "Book"
    return f"""You are helping build a cross-domain recommendation system that matches {domain_name}s to users across different media types.

Given the {domain_name} title "{title}", generate a detailed semantic profile covering:

1. Genre & Subgenre: Be specific (e.g. "neo-noir psychological thriller" not just "thriller")
2. Plot & Narrative: Core story arc, narrative structure (linear, nonlinear, episodic)
3. Style & Tone: Visual/descriptive style, pacing, atmosphere
4. Themes: Central themes and motifs (e.g. redemption, class conflict, coming-of-age)
5. Audience & Mood: Target audience, emotional experience
6. Similar To: Mention a few well-known works with similar appeal (cross-domain if relevant)

Write 4-6 natural sentences. Be concrete, avoid vague marketing language. Every word should help distinguish this item from similar ones. Keep under {MAX_WORDS} words total.

Output only the profile text, nothing else."""

In [3]:
def call_deepseek(prompt):
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": "You are a professional item analyst. Output ONLY the profile text, no prefix, no quotation marks."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
                extra_body={"thinking": {"type": "disabled"}}
            )
            text = response.choices[0].message.content.strip()
            return text[:MAX_CHARS]
        except Exception as e:
            err = str(e)
            if "rate_limit" in err.lower() or "429" in err:
                time.sleep(SLEEP_RATE_LIMIT)
            elif attempt < 2:
                time.sleep(2)
            else:
                print(f"  API fail (attempt {attempt+1}): {err[:100]}")
    return ""

In [4]:
# 主流程（多线程 + 断点续传)
print("=" * 60)
print(f"LLM 文本增强 V2 — {MAX_WORDS} 词 / {NUM_THREADS} 线程")
print("=" * 60)

meta_df = pd.read_csv(META_CSV)
print(f"总物品数: {len(meta_df)}")

# 断点续传
if os.path.exists(OUTPUT_CSV):
    saved = pd.read_csv(OUTPUT_CSV)
    done_mask = saved["llm_enhanced_text"].notna() & (saved["llm_enhanced_text"] != "")
    if done_mask.any():
        meta_df = pd.merge(
            meta_df,
            saved[["parent_asin", "llm_enhanced_text"]].loc[done_mask],
            on="parent_asin", how="left"
        )
        print(f"断点续传：已有 {done_mask.sum()} 条完成")
    else:
        meta_df["llm_enhanced_text"] = ""
else:
    meta_df["llm_enhanced_text"] = ""

to_process = meta_df[
    meta_df["llm_enhanced_text"].isna() | (meta_df["llm_enhanced_text"] == "")
]
print(f"待处理: {len(to_process)}")

if len(to_process) == 0:
    print("全部已完成！")
else:
    tasks = [(idx, row["domain"], row["title"]) for idx, row in to_process.iterrows()]
    completed = 0

    def process_one(task):
        idx, domain, title = task
        prompt = build_prompt(domain, title)
        enhanced = call_deepseek(prompt)
        return idx, enhanced

    with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
        futures = {executor.submit(process_one, t): t for t in tasks}
        for future in tqdm(as_completed(futures), total=len(futures), desc="V2 生成"):
            idx, enhanced = future.result()
            meta_df.at[idx, "llm_enhanced_text"] = enhanced
            completed += 1
            if completed % BATCH_SAVE == 0:
                with write_lock:
                    meta_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

    meta_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    print(f"完成 → {OUTPUT_CSV}")

LLM 文本增强 V2 — 200 词 / 20 线程
总物品数: 43528
待处理: 43528


V2 生成: 100%|██████████| 43528/43528 [2:07:29<00:00,  5.69it/s]  


完成 → final/item_meta_llm_enhanced_v2.csv


### 结果预览

In [ ]:
df = pd.read_csv(OUTPUT_CSV)
has = df["llm_enhanced_text"].notna() & (df["llm_enhanced_text"] != "")
print(f"增强完成: {has.sum()}/{len(df)}")

llm_chars = df["llm_enhanced_text"].fillna("").str.len()
llm_words = df["llm_enhanced_text"].fillna("").str.split().str.len()
print(f"\n字符数: avg={llm_chars.mean():.0f} median={llm_chars.median():.0f} min={llm_chars.min():.0f} max={llm_chars.max():.0f}")
print(f"词数:   avg={llm_words.mean():.0f} median={llm_words.median():.0f} min={llm_words.min():.0f} max={llm_words.max():.0f}")

print("\n--- 样例 ---")
for _, r in df[has].sample(min(3, has.sum()), random_state=1).iterrows():
    tag = "[movie]" if r["domain"] == 0 else "[book]"
    print(f"\n{tag} {r['title'][:80]}")
    print(f"  ({len(r['llm_enhanced_text'])} chars, {len(r['llm_enhanced_text'].split())} words)")
    print(f"  {r['llm_enhanced_text'][:400]}")

增强完成: 43528/43528

字符数: avg=850 median=848 min=417 max=1398
词数:   avg=125 median=124 min=63 max=201

--- 样例 ---

[movie] Sense & Sensibility / Persuasion Collector's Set (Includes Miss Austen Regrets)
  (931 chars, 136 words)
  This collector's set combines three period dramas centered on the romantic and social constraints of 19th-century England, specifically the works of Jane Austen. The narratives are linear and character-driven, focusing on the emotional journeys of women navigating love, family duty, and financial security within strict societal hierarchies. The tone is a blend of gentle satire and earnest romance,

[book] The Wet Nurse's Tale
  (868 chars, 126 words)
  A historical fiction novel set in Victorian England, "The Wet Nurse's Tale" follows a lower-class woman who becomes a wet nurse for an aristocratic family, navigating the intimate yet exploitative dynamics of class and motherhood. The narrative is linear, driven by her personal struggle for agency within a rigid s